In [1]:
!mkdir working

In [3]:
import kagglehub
kagglehub.login()


In [4]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

jan_2026_dl_gen_ai_project_path = kagglehub.competition_download('jan-2026-dl-gen-ai-project')

print('Data source import complete.')


100%|██████████| 16.1G/16.1G [12:37<00:00, 22.9MB/s]  

Extracting files...


Data source import complete.


In [ ]:
import torch
import numpy as np
import random
import torchaudio
import os
import glob
from pathlib import Path

# --- SET YOUR KAGGLE PATHS ---
INPUT_BASE = '/root/.cache/kagglehub/competitions/jan-2026-dl-gen-ai-project/messy_mashup'
WORKING_BASE = '/content/working'

STEMS_PATH = os.path.join(INPUT_BASE, 'genres_stems')
NOISE_PATH = os.path.join(INPUT_BASE, 'ESC-50-master/audio')
OUTPUT_PATH = os.path.join(WORKING_BASE, 'synthetic_mashups/train')


def seed_everything(seed=42):
    """Locks all random seeds for absolute reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    
    # If using GPU
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        # Forces deterministic algorithms
        torch.backends.cudnn.deterministic = True 
        torch.backends.cudnn.benchmark = False

# Execute immediately at the top of the script
seed_everything(42)


def generate_synthetic_dataset(stems_dir, noise_dir, output_dir, samples_per_genre=50, target_sr=22050, duration=30):
    """Generates deterministic noisy mashups and saves them to /kaggle/working/."""
    genres = ["blues", "classical", "country", "disco", "hiphop",
"jazz", "metal", "pop", "reggae", "rock"
]
    target_length = target_sr * duration
    
    # Get noise files from read-only input
    noise_files = glob.glob(os.path.join(noise_dir, '**', '*.wav'), recursive=True)
    
    for genre in genres:
        # Create output directories in the writable /kaggle/working/ directory
        genre_out_dir = Path(output_dir) / genre
        genre_out_dir.mkdir(parents=True, exist_ok=True)
        
        song_folders = glob.glob(os.path.join(stems_dir, genre, '*'))
        if not song_folders: 
            print(f"Warning: No songs found for genre {genre}")
            continue
        
        for i in range(samples_per_genre):
            chosen_songs = random.sample(song_folders, 4)
            stems = []
            stem_types = ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']
            
            for song, stem_type in zip(chosen_songs, stem_types):
                stem_path = os.path.join(song, stem_type)
                if os.path.exists(stem_path):
                    waveform, sr = torchaudio.load(stem_path)
                    
                    # Basic Resampling check (if needed)
                    if sr != target_sr:
                        resampler = torchaudio.transforms.Resample(sr, target_sr)
                        waveform = resampler(waveform)

                    if waveform.shape[1] > target_length:
                        waveform = waveform[:, :target_length]
                    elif waveform.shape[1] < target_length:
                        waveform = torch.nn.functional.pad(waveform, (0, target_length - waveform.shape[1]))
                    stems.append(waveform)
            
            if len(stems) == 4:
                mashup = torch.stack(stems).sum(dim=0)
                mashup = mashup / (torch.max(torch.abs(mashup)) + 1e-8)
                
                noise_file = random.choice(noise_files)
                noise, _ = torchaudio.load(noise_file)
                
                if noise.shape[1] > target_length:
                    noise = noise[:, :target_length]
                    
                start_idx = random.randint(0, target_length - noise.shape[1])
                intensity = random.uniform(0.1, 0.4)
                
                mashup[:, start_idx:start_idx + noise.shape[1]] += (noise * intensity)
                mashup = mashup / (torch.max(torch.abs(mashup)) + 1e-8)
                
                # Save to /kaggle/working/
                out_path = genre_out_dir / f"mashup_{i:03d}.wav"
                torchaudio.save(str(out_path), mashup, target_sr)

# Run the generation
generate_synthetic_dataset(STEMS_PATH, NOISE_PATH, OUTPUT_PATH, samples_per_genre=50)




import os
import glob
import torch
import torchaudio
from pathlib import Path

def extract_and_save_features(input_dir, output_dir, target_sr=22050):
    """Converts audio to Mel-spectrograms in dB and saves as PyTorch tensors."""
    mel_transform = torchaudio.transforms.MelSpectrogram(
        sample_rate=target_sr, n_fft=2048, hop_length=512, n_mels=128
    )
    amplitude_to_db = torchaudio.transforms.AmplitudeToDB()

    # Find all .wav files in the input directory
    wav_files = glob.glob(os.path.join(input_dir, '**', '*.wav'), recursive=True)
    
    if not wav_files:
        print(f"Warning: No .wav files found in {input_dir}")
        return

    for wav_path in wav_files:
        # Replicate directory structure
        rel_path = os.path.relpath(wav_path, input_dir)
        out_path = Path(output_dir) / rel_path
        out_path = out_path.with_suffix('.pt')
        
        # Ensure the target directory exists in /kaggle/working/
        out_path.parent.mkdir(parents=True, exist_ok=True)
        
        # Process and save
        waveform, sr = torchaudio.load(wav_path)
        mel_spec = mel_transform(waveform)
        mel_spec_db = amplitude_to_db(mel_spec)
        
        torch.save(mel_spec_db, out_path)
    
    print(f"Successfully saved {len(wav_files)} feature files to {output_dir}")


INPUT_DIR = '/content/working/synthetic_mashups/train'
OUTPUT_DIR = '/content/working/features/train'

extract_and_save_features(INPUT_DIR, OUTPUT_DIR)

## Q.1

In [8]:
import os

train_dir = "/content/working/synthetic_mashups/train"

count = 0
for root, dirs, files in os.walk(train_dir):
    for name in files:
        if name.lower().endswith(".wav"):
            count += 1

print("Total .wav files:", count)


Total .wav files: 500


## Q.2

In [12]:
y,sr=torchaudio.load("/content/working/synthetic_mashups/train/blues/mashup_000.wav")
y.shape

torch.Size([2, 661500])

## Q.3

In [15]:
y=torch.load("/content/working/features/train/blues/mashup_000.pt")
y.shape

torch.Size([2, 128, 1292])

In [43]:
import torch
import torch.nn as nn
import glob
import os
from pathlib import Path
from torch.utils.data import Dataset, DataLoader

class PrecomputedFeatureDataset(Dataset):
    def __init__(self, features_dir):
        self.files = glob.glob(os.path.join(features_dir, '**', '*.pt'), recursive=True)
        self.genres = sorted(['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock'])
        self.genre_to_idx = {g: i for i, g in enumerate(self.genres)}

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        file_path = self.files[idx]
        # Extract genre from the directory name
        genre = Path(file_path).parent.name
        label = self.genre_to_idx[genre] # converts genre name to a numerically encoded value
        
        # Load precomputed tensor
        feature = torch.load(file_path)
        if feature.shape[0] > 1:
            feature = feature.mean(dim=0, keepdim=True)
        
        return feature, label

class CRNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        
        # TODO 1: Define the CNN Backbone using nn.Sequential
        # Expect an input of shape (Batch_Size, 1, 128, Time_Steps)
        # Block 1: Conv2d(1 -> 32, kernel=3, padding=1) -> BatchNorm2d -> ReLU -> MaxPool2d(2)
        # Block 2: Conv2d(32 -> 64, kernel=3, padding=1) -> BatchNorm2d -> ReLU -> MaxPool2d(2)
        self.cnn=nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2),
            )
        
        # TODO 2: Define the RNN Component
        # Hint: Calculate the flattened feature size coming out of your CNN.
        # Original Mels = 128. After two MaxPool2d(2) layers, Mels = 128 / 4 = 32.
        # Channels = 64. So, Input Size = Channels * Mels = 2048.
        # Create a 1-layer Bidirectional LSTM with hidden_size=64 and batch_first=True.
        self.lstm=nn.LSTM(
            input_size=2048,
            hidden_size=64,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
            )
        
        # TODO 3: Define the Classifier
        # Create a Fully Connected (Linear) layer. 
        # Hint: Since the LSTM is bidirectional, the input features will be hidden_size * 2.

        self.fc = nn.Linear(128, num_classes)

    def forward(self, x):
        """
        Input:
            x: Tensor of shape (Batch, 1, 128, Time) representing Mel-spectrograms.
        Output:
            logits: Tensor of shape (Batch, num_classes) representing unnormalized class scores.
        """
        
        # TODO 4: Pass 'x' through the CNN backbone
        # Expected shape after CNN: (Batch, Channels=64, Mels=32, Time)

        x = self.cnn(x)

        # TODO 5: Bridge the gap (Shape Manipulation)
        # Permute and reshape the CNN output to be compatible with the LSTM.
        # Extract b, c, f, t from the tensor shape.
        # Permute the dimensions to bring Time forward, then reshape to flatten Channels and Mels.
        # Target shape for LSTM: (Batch, Time_Steps, Channels * Mels)

        b, c, f, t = x.shape
        # print(x.shape)
        x = x.permute(0, 3, 1, 2)
        x = x.reshape(b, t, c * f)
        
        # TODO 6: Pass the reshaped sequence through the LSTM
        # The LSTM returns output and (hidden_state, cell_state). You only need the output.

        lstm_out, _ = self.lstm(x)
        
        # TODO 7: Global Max Pooling over the time dimension
        # Collapse the sequence down to a single vector using torch.max() over the time dimension (dim=1).
        # Note: torch.max returns both values and indices. You only need the values.

        pooled, _ = torch.max(lstm_out, dim=1)
        
        # TODO 8: Pass the pooled vector through the fully connected layer
        
        logits = self.fc(pooled)

        # return logits
        return logits
        raise NotImplementedError("You need to implement the CRNN forward pass!")

## Q.4

In [44]:
model = CRNN()
dummy = torch.randn(32, 1, 128, 1292)  
x = model.cnn(dummy)
print(tuple(x.shape))

(32, 64, 32, 323)


## Q.5

In [45]:
sum(p.numel() for p in model.lstm.parameters() if p.requires_grad)

1082368

## Model Training


In [46]:
dataset=PrecomputedFeatureDataset('/content/working/features/train')

In [47]:
from torch.utils.data import random_split

train_data,val_data=random_split(dataset,[int(0.8*len(dataset)),int(0.2*len(dataset))])


In [48]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CRNN(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)



train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32, shuffle=False)

num_epochs = 10

In [49]:
for epoch in range(num_epochs):
    # ----- TRAIN -----
    model.train()
    running_loss = 0.0
    running_correct = 0
    running_total = 0

    for inputs, targets in train_loader:
        inputs = inputs.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        _, preds = outputs.max(1)
        running_correct += preds.eq(targets).sum().item()
        running_total += targets.size(0)

    train_loss = running_loss / running_total
    train_acc = running_correct / running_total

    # ----- VALIDATION -----
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs = inputs.to(device)
            targets = targets.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, targets)

            val_loss += loss.item() * inputs.size(0)
            _, preds = outputs.max(1)
            val_correct += preds.eq(targets).sum().item()
            val_total += targets.size(0)

    val_loss /= val_total
    val_acc = val_correct / val_total

    print(
        f"Epoch {epoch+1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}"
    )

Epoch 1/10 | Train Loss: 2.2310 | Train Acc: 0.2200 | Val Loss: 2.1285 | Val Acc: 0.3400
Epoch 2/10 | Train Loss: 1.9673 | Train Acc: 0.3750 | Val Loss: 1.9089 | Val Acc: 0.4600
Epoch 3/10 | Train Loss: 1.8015 | Train Acc: 0.5000 | Val Loss: 1.8203 | Val Acc: 0.4600
Epoch 4/10 | Train Loss: 1.6675 | Train Acc: 0.5225 | Val Loss: 1.7350 | Val Acc: 0.4800
Epoch 5/10 | Train Loss: 1.5714 | Train Acc: 0.5950 | Val Loss: 1.5994 | Val Acc: 0.5500
Epoch 6/10 | Train Loss: 1.4568 | Train Acc: 0.6125 | Val Loss: 1.5584 | Val Acc: 0.5200
Epoch 7/10 | Train Loss: 1.3582 | Train Acc: 0.6900 | Val Loss: 1.4333 | Val Acc: 0.6400
Epoch 8/10 | Train Loss: 1.2582 | Train Acc: 0.6975 | Val Loss: 1.4007 | Val Acc: 0.6100
Epoch 9/10 | Train Loss: 1.1833 | Train Acc: 0.7675 | Val Loss: 1.3477 | Val Acc: 0.5800
Epoch 10/10 | Train Loss: 1.1151 | Train Acc: 0.7925 | Val Loss: 1.2934 | Val Acc: 0.5900
